# MyPhatshala · Kafka in Colab: 2 real-time projects
Runs Apache Kafka 4.4 (KRaft, single node) inside this Colab machine, then two Python projects.
1. **TiffinGo live order tracker** (producer → topic `orders` → live dashboard consumer)
2. **UPI fraud alert** (payments → fraud detector → `fraud-alerts` → SMS notifier)

Run the cells top to bottom. Kafka lives only as long as this Colab session.

In [ ]:
# 1) Java 17 (Kafka 4.x needs Java 17+) and the Python client
!apt-get -qq install -y openjdk-17-jre-headless > /dev/null
!java -version
!pip -q install confluent-kafka

In [ ]:
# 2) Download and unpack Kafka 4.4.0 (archive.apache.org keeps old versions)
!wget -q https://downloads.apache.org/kafka/4.4.0/kafka_2.13-4.4.0.tgz || wget -q https://archive.apache.org/dist/kafka/4.4.0/kafka_2.13-4.4.0.tgz
!tar -xzf kafka_2.13-4.4.0.tgz
%cd kafka_2.13-4.4.0

In [ ]:
# 3) Format storage (KRaft) and start Kafka in the background
import subprocess, time
cid = subprocess.check_output(['bin/kafka-storage.sh', 'random-uuid']).decode().split()[-1]
!bin/kafka-storage.sh format --standalone -t {cid} -c config/server.properties
kafka = subprocess.Popen(['bin/kafka-server-start.sh', 'config/server.properties'],
                         stdout=open('kafka.log', 'w'), stderr=subprocess.STDOUT)   # keeps running
time.sleep(15)
!grep -o 'Kafka Server started' kafka.log

In [ ]:
# 4) Create the three topics
!bin/kafka-topics.sh --bootstrap-server localhost:9092 --create --topic orders --partitions 3
!bin/kafka-topics.sh --bootstrap-server localhost:9092 --create --topic payments --partitions 3
!bin/kafka-topics.sh --bootstrap-server localhost:9092 --create --topic fraud-alerts --partitions 1

## Project 1 · TiffinGo live order tracker
We save the two scripts, start the producer in the background, and run the dashboard here.

In [ ]:
%%writefile p1_order_producer.py
"""Project 1 · TiffinGo live order tracker: producer. Sends one order every 0.2 s."""
import json, random, time, sys
from confluent_kafka import Producer
random.seed(7)
N = int(sys.argv[1]) if len(sys.argv) > 1 else 50
p = Producer({"bootstrap.servers": "localhost:9092", "partitioner": "murmur2_random"})
cities = ["Mumbai", "Pune", "Delhi", "Bengaluru"]
for i in range(1, N + 1):
    order = {"order_id": f"o{i:04d}", "city": random.choice(cities),
             "amount": random.choice([120, 180, 250, 320, 480, 1500]), "ts": time.time()}
    p.produce("orders", key=order["city"], value=json.dumps(order))
    p.poll(0)
    time.sleep(0.2)
p.flush()
print(f"sent {N} orders")


In [ ]:
%%writefile p1_live_dashboard.py
"""Project 1 · TiffinGo live order tracker: consumer. Running revenue per city + big-order alert."""
import json, time
from collections import defaultdict
from confluent_kafka import Consumer
c = Consumer({"bootstrap.servers": "localhost:9092", "group.id": "live-dashboard", "auto.offset.reset": "earliest"})
c.subscribe(["orders"])
revenue, count, seen, idle = defaultdict(int), defaultdict(int), 0, 0
while idle < 10:                                   # stop after 10 s with no data (demo)
    m = c.poll(1.0)
    if m is None or m.error():
        idle += 1; continue
    idle = 0
    o = json.loads(m.value()); seen += 1
    revenue[o["city"]] += o["amount"]; count[o["city"]] += 1
    if o["amount"] >= 1000:
        print(f"  ALERT big order {o['order_id']} {o['city']} Rs {o['amount']} (partition {m.partition()}, offset {m.offset()})")
    if seen % 10 == 0:
        board = "  ".join(f"{k}: {count[k]} / Rs {revenue[k]}" for k in sorted(revenue))
        print(f"[{seen:3} orders] {board}")
c.close()


In [ ]:
producer = subprocess.Popen(['python', 'p1_order_producer.py', '50'])   # sends 1 order every 0.2 s
!python p1_live_dashboard.py

In [ ]:
# Lag per partition for the dashboard group
!bin/kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group live-dashboard

## Project 2 · UPI fraud alert pipeline

In [ ]:
%%writefile p2_payment_producer.py
"""Project 2 · UPI fraud alert: simulated payments, including one card that pays very fast."""
import json, random, time
T0 = 1793900000.0                                   # simulated clock: one payment every 5 s
from confluent_kafka import Producer
random.seed(11)
p = Producer({"bootstrap.servers": "localhost:9092", "partitioner": "murmur2_random"})
users = [f"user{i}@upi" for i in range(1, 9)]
for i in range(1, 61):
    u = random.choice(users)
    if 30 <= i < 36: u = "user9@upi"              # 6 rapid payments from one account
    amt = random.choice([50, 120, 300, 900]) if u != "user9@upi" else 4999
    if i == 45: u, amt = "user3@upi", 75000         # one unusually large payment
    p.produce("payments", key=u, value=json.dumps({"txn": f"t{i:03d}", "vpa": u, "amount": amt, "ts": T0 + (i * 5 if not 30 <= i < 36 else 150 + (i - 30) * 3)}))
    p.poll(0); time.sleep(0.1)
p.flush(); print("sent 60 payments")


In [ ]:
%%writefile p2_fraud_detector.py
"""Project 2 · UPI fraud alert: reads payments, applies 2 rules, writes alerts to another topic."""
import json, time
from collections import defaultdict, deque
from confluent_kafka import Consumer, Producer
c = Consumer({"bootstrap.servers": "localhost:9092", "group.id": "fraud-detector", "auto.offset.reset": "earliest"})
p = Producer({"bootstrap.servers": "localhost:9092"})
c.subscribe(["payments"])
recent = defaultdict(deque)                          # vpa -> timestamps in the last 60 s
checked, alerts, idle = 0, 0, 0
while idle < 8:
    m = c.poll(1.0)
    if m is None or m.error():
        idle += 1; continue
    idle = 0; checked += 1
    t = json.loads(m.value()); q = recent[t["vpa"]]
    q.append(t["ts"])
    while q and t["ts"] - q[0] > 60: q.popleft()
    reason = "amount over Rs 50,000" if t["amount"] > 50000 else ("more than 4 payments in 60 s" if len(q) > 4 else None)
    if reason:
        alerts += 1
        p.produce("fraud-alerts", key=t["vpa"], value=json.dumps({**t, "reason": reason}))
        print(f"  FLAG {t['txn']} {t['vpa']} Rs {t['amount']}: {reason}")
p.flush(); c.close()
print(f"checked {checked} payments, raised {alerts} alerts")


In [ ]:
%%writefile p2_notifier.py
"""Project 2 · UPI fraud alert: a second service that reads the alerts topic and 'sends SMS'."""
import json
from confluent_kafka import Consumer
c = Consumer({"bootstrap.servers": "localhost:9092", "group.id": "sms-notifier", "auto.offset.reset": "earliest"})
c.subscribe(["fraud-alerts"]); idle = 0
while idle < 6:
    m = c.poll(1.0)
    if m is None or m.error():
        idle += 1; continue
    idle = 0; a = json.loads(m.value())
    print(f"  SMS to {a['vpa']}: payment {a['txn']} of Rs {a['amount']} held for review ({a['reason']})")
c.close()


In [ ]:
!python p2_payment_producer.py
!python p2_fraud_detector.py
!python p2_notifier.py

## Try next
- Change the fraud rule (window, count, amount) and re-run the detector with a new `group.id` to re-read all payments.
- Run two dashboards with the same group id in two cells: they split the partitions.
- Stop Kafka: `!bin/kafka-server-stop.sh`